# SMS Spam Classification using SVM (Leak-Free Training & 5-Fold Cross Validation)
Dataset: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?resource=download

In [1]:
# Install (if needed)
!pip -q install wordcloud nltk scikit-learn


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
import re, string, nltk
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score,classification_report,
confusion_matrix,ConfusionMatrixDisplay,
precision_score,recall_score,f1_score,roc_curve,auc)
from sklearn.utils import resample

nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Upload dataset (`spam.csv`) downloaded from Kaggle.

In [3]:
from google.colab import files
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    pass


Saving spam.csv to spam.csv


In [4]:
df = pd.read_csv("spam.csv", encoding="latin-1")

df = df[['v1', 'v2']]

df.columns = ['label', 'message']


In [5]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print("Duplicates:",df.duplicated().sum())
print(df['label'].value_counts())


(5572, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None
label      0
message    0
dtype: int64
Duplicates: 403
label
ham     4825
spam     747
Name: count, dtype: int64


## Exploratory Data Analysis

In [6]:
df['char_count']=df['message'].str.len()
df['word_count']=df['message'].str.split().str.len()

fig,ax=plt.subplots(1,2,figsize=(10,4))
sns.countplot(data=df,x='label',ax=ax[0])
sns.histplot(data=df,x='word_count',hue='label',bins=30,ax=ax[1])
plt.tight_layout()
plt.show()


<Figure size 1000x400 with 2 Axes>

In [7]:
spam=' '.join(df[df.label=='spam'].message)
ham=' '.join(df[df.label=='ham'].message)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(WordCloud(background_color='white').generate(spam))
plt.axis('off'); plt.title("Spam")
plt.subplot(1,2,2)
plt.imshow(WordCloud(background_color='white').generate(ham))
plt.axis('off'); plt.title("Ham")
plt.show()


<Figure size 1200x500 with 2 Axes>

## Preprocessing

In [8]:
# Load the list of English stopwords (e.g., "the", "is", "and")
stop_words = set(stopwords.words('english'))

# Create a WordNet lemmatizer object to convert words to their base form
lemm = WordNetLemmatizer()

# Define a function to clean and preprocess SMS text
def clean_text(text):

    # Convert all characters to lowercase
    text = str(text).lower()

    # Remove URLs (e.g., http://example.com or www.example.com)
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove email addresses (e.g., abc@gmail.com)
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove all numeric digits from the text
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation marks such as !, ?, ., ,, :, etc.
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Split the sentence into words,
    # remove stopwords,
    # and convert each word to its base (lemma) form
    tokens = [
        lemm.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]

    # Join the cleaned words back into a single string
    return ' '.join(tokens)

# Remove duplicate SMS messages from the dataset
df = df.drop_duplicates()

# Apply the clean_text() function to every message
# and store the cleaned text in a new column named 'clean_message'
df['clean_message'] = df['message'].apply(clean_text)

# Convert categorical labels into numerical values
# ham  -> 0
# spam -> 1
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

# Display the first five rows of the processed dataset
df.head()


   label                                                                                                                                                      message                                                                                          clean_message
0      0                                              Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                     go jurong point crazy available bugis n great world la e buffet cine got amore wat
1      0                                                                                                                                Ok lar... Joking wif u oni...                                                                                ok lar joking wif u oni
2      1  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's  free entry wkly comp win f

## Train/Test Split (Leak-Free Workflow)
To prevent **Data Leakage**, we split the original dataset into Training (80%) and Testing (20%) **BEFORE** applying oversampling. The test set remains 100% untouched and unseen.

In [9]:
X = df['clean_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Raw Training Set Shape: {X_train.shape} (Ham: {sum(y_train==0)}, Spam: {sum(y_train==1)})")
print(f"Pure Testing Set Shape : {X_test.shape} (Ham: {sum(y_test==0)}, Spam: {sum(y_test==1)})")


Raw Training Set Shape: (4135,) (Ham: 3613, Spam: 522)
Pure Testing Set Shape : (1034,) (Ham: 903, Spam: 131)


## Balancing Training Set (Oversampling Spam Data)
We oversample the minority class (Spam) **ONLY within the Training Set** (3,613 Ham vs 522 Spam $\rightarrow$ 3,613 Ham vs 3,613 Spam). This balances the model's learning while avoiding data leakage into the test set.

In [10]:
# Oversample spam in training set only
train_df = pd.DataFrame({'clean_message': X_train, 'label': y_train})
train_ham = train_df[train_df.label == 0]
train_spam = train_df[train_df.label == 1]

train_spam_oversampled = resample(train_spam, replace=True, n_samples=len(train_ham), random_state=42)
train_balanced = pd.concat([train_ham, train_spam_oversampled]).reset_index(drop=True)

X_train_bal = train_balanced['clean_message']
y_train_bal = train_balanced['label']

print("Training set class distribution before oversampling:")
print(y_train.value_counts())
print("\nTraining set class distribution after oversampling:")
print(y_train_bal.value_counts())

# Plot comparison of training set before vs after oversampling
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=train_df, x='label', ax=axes[0], hue='label', palette=['#4C72B0', '#DD8452'], legend=False)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Ham (3,613)', 'Spam (522)'])
axes[0].set_title('Training Set Before Oversampling', fontweight='bold')

sns.countplot(data=train_balanced, x='label', ax=axes[1], hue='label', palette=['#4C72B0', '#DD8452'], legend=False)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Ham (3,613)', 'Spam (3,613)'])
axes[1].set_title('Training Set After Oversampling', fontweight='bold')
plt.tight_layout()
plt.show()


Training set class distribution before oversampling:
label
0    3613
1     522

Training set class distribution after oversampling:
label
0    3613
1    3613


<Figure size 1200x450 with 2 Axes>

## SVM Model

In [11]:
pipeline=Pipeline([
('tfidf',TfidfVectorizer(max_features=5000,ngram_range=(1,2))),
('svm',LinearSVC())
])

pipeline.fit(X_train_bal,y_train_bal)
pred=pipeline.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print("Precision:",precision_score(y_test,pred))
print("Recall:",recall_score(y_test,pred))
print("F1:",f1_score(y_test,pred))
print(classification_report(y_test,pred))


Accuracy: 0.9777562862669246
Precision: 0.9153846153846154
Recall: 0.9083969465648855
F1: 0.9118773946360154
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       903
           1       0.92      0.91      0.91       131

    accuracy                           0.98      1034
   macro avg       0.95      0.95      0.95      1034
weighted avg       0.98      0.98      0.98      1034



## Hyperparameter Tuning

In [12]:
param_grid={
'svm__C':[0.1,1,10]
}
grid=GridSearchCV(pipeline,param_grid,cv=5)
grid.fit(X_train_bal,y_train_bal)

print("Best Parameters:",grid.best_params_)
print("Best CV Score:",grid.best_score_)


Best Parameters: {'svm__C': 10}
Best CV Score: 0.9962634543688111


## 5-Fold Cross Validation

In [13]:
scores=cross_val_score(grid.best_estimator_,X_train_bal,y_train_bal,cv=5)
print(scores)
print("Mean Accuracy:",scores.mean())


[0.99654219 0.99653979 0.99307958 0.99653979 0.99861592]
Mean Accuracy: 0.9962634543688111


## Prediction Function

In [14]:
model=grid.best_estimator_

def predict_sms(text):
    pred=model.predict([clean_text(text)])[0]
    return "Spam" if pred==1 else "Ham"

print(predict_sms("Congratulations! You have won a free iPhone. Click here now."))
print(predict_sms("Hi, are we meeting tomorrow at 3 PM?"))


Spam
Ham
